# Citation Index API: executable end-to-end guide

This notebook is both a tutorial and a final smoke test for the Docker deployment. It runs the complete supported workflow:

1. upload a PDF to the external **MinerU** extractor,
2. ask **Qwen3.6** to identify the bibliography entries, and
3. ask **Qwen3.6** to convert those entries into structured records.

Each API operation is asynchronous: submission returns a `job_id`, the client polls `/jobs/{job_id}/status`, and the result is read from `/jobs/{job_id}`. The notebook intentionally fails fast if a service is unavailable or a stage returns an invalid result.

> The combined `/process/references` route is not currently enabled, so this notebook demonstrates the supported three-stage composition explicitly.

## Goal

A successful top-to-bottom run proves all of the following:

- the API, Redis, storage, and RQ workers are available;
- Docker containers can reach the host-side MinerU port-forward;
- prompt files are present in the image;
- Qwen returns answer content and valid structured JSON; and
- job failures and missing job IDs are surfaced as errors rather than empty successes.

## Setup

Start MinerU in one terminal. This command remains attached while the port-forward is active:

```bash
oc port-forward service/mineru-api 8000:8000
```

Start the lightweight Docker stack in a second terminal from the repository root:

```bash
docker compose up -d --build --scale worker-llm=1 \
  redis api worker-default worker-llm
docker compose ps
```

The Citation Index API is published on host port **8001** because port **8000** is reserved for MinerU. Docker Compose reads the Qwen endpoint and API key from `.env`; never put credentials in this notebook.

In [1]:
import json
import os
import time
import uuid
from pathlib import Path

import requests

### Configuration

Defaults target the local Docker stack and a small repository fixture. Override either value through environment variables when needed:

```bash
export CITATION_INDEX_API=http://localhost:8001
export CITATION_INDEX_PDF=/absolute/path/to/paper.pdf
```

In [2]:
API_BASE = os.getenv("CITATION_INDEX_API", "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl/").rstrip("/")
PDF_PATH = Path(
    os.getenv("CITATION_INDEX_PDF", "../benchmarks/excite/all_pdfs/44404.pdf")
).expanduser().resolve()

POLL_INTERVAL_SECONDS = 2
MAX_JOB_WAIT_SECONDS = 1_800
HTTP_TIMEOUT = (10, 120)  # connect timeout, response timeout

session = requests.Session()
print({"api": API_BASE, "pdf": str(PDF_PATH), "pdf_exists": PDF_PATH.is_file()})
assert PDF_PATH.is_file(), f"PDF not found: {PDF_PATH}"

{'api': 'https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl', 'pdf': '/Users/alex/docs/code/Odoma/citation_index/benchmarks/excite/all_pdfs/44404.pdf', 'pdf_exists': True}


## Step 1 — Verify service health

`GET /health` checks the API's Redis and storage dependencies. It does not call MinerU or Qwen, so those integrations are tested by the later stages.

In [3]:
health_response = session.get(f"{API_BASE}/health", timeout=HTTP_TIMEOUT)
health_response.raise_for_status()
health = health_response.json()
print(health)
assert health == {"status": "healthy", "redis": "ok", "storage": "ok", "version": "0.2.0"}

{'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '0.2.0'}


## Step 2 — Understand the asynchronous job contract

Submission endpoints return HTTP 200 with a job object such as:

```json
{"job_id": "...", "status": "queued", "created_at": "..."}
```

A job then moves through `queued` and `processing` to either `completed` or `failed`. The helper below prints only state changes, raises with the server error on failure, and enforces a maximum wait.

In [4]:
def response_json(response: requests.Response) -> dict:
    """Raise an informative HTTP error, then return a JSON object."""
    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        preview = response.text[:300].replace("\n", " ")
        raise RuntimeError(
            f"Expected JSON from {response.request.method} {response.url}; "
            f"HTTP {response.status_code}, body={preview!r}"
        ) from exc
    if not response.ok:
        raise RuntimeError(
            f"{response.request.method} {response.url} failed with "
            f"HTTP {response.status_code}: {payload}"
        )
    if not isinstance(payload, dict):
        raise TypeError(f"Expected a JSON object, got {type(payload).__name__}")
    return payload


def wait_for_job(job_id: str, max_wait: int = MAX_JOB_WAIT_SECONDS) -> dict:
    """Poll a job to a terminal state and return its persisted result."""
    started = time.monotonic()
    last_state = None

    while time.monotonic() - started < max_wait:
        status_response = session.get(
            f"{API_BASE}/jobs/{job_id}/status", timeout=HTTP_TIMEOUT
        )
        status_payload = response_json(status_response)
        state = status_payload["status"]

        if state != last_state:
            elapsed = time.monotonic() - started
            stage = status_payload.get("current_stage")
            print(f"[{elapsed:6.1f}s] {job_id}: {state} (stage={stage})")
            last_state = state

        if state == "completed":
            return response_json(
                session.get(f"{API_BASE}/jobs/{job_id}", timeout=HTTP_TIMEOUT)
            )
        if state == "failed":
            raise RuntimeError(
                f"Job {job_id} failed: {status_payload.get('error', 'unknown error')}"
            )

        time.sleep(POLL_INTERVAL_SECONDS)

    raise TimeoutError(f"Job {job_id} did not finish within {max_wait}s")

## Step 3 — Extract Markdown with MinerU

`POST /extract/text` accepts a PDF as multipart form data. Supported extractor names are `pymupdf`, `mineru`, and `grobid`; Marker has been removed. This test selects `mineru`, which makes a real call through `host.docker.internal:8000` to the OpenShift port-forward.

In [5]:
with PDF_PATH.open("rb") as pdf_file:
    submit_text_response = session.post(
        f"{API_BASE}/extract/text",
        params={"extractor": "mineru", "markdown": "true"},
        files={"file": (PDF_PATH.name, pdf_file, "application/pdf")},
        timeout=HTTP_TIMEOUT,
    )

text_job = response_json(submit_text_response)
print(text_job)
assert text_job["status"] in {"queued", "processing"}

{'job_id': '5f6c5b21-ffe9-48ee-9104-8d031ac2a159', 'status': 'queued', 'created_at': '2026-07-22T16:03:58.275911', 'message': 'Text extraction job enqueued'}


In [6]:
text_result = wait_for_job(text_job["job_id"])
extracted_text = text_result.get("text", "")

assert text_result.get("extractor") == "mineru", text_result
assert len(extracted_text) > 1_000, "MinerU returned unexpectedly little text"

print(
    {
        "extractor": text_result.get("extractor"),
        "backend": text_result.get("backend"),
        "mineru_version": text_result.get("mineru_version"),
        "text_characters": len(extracted_text),
    }
)
print("\nPreview:\n", extracted_text[:700].strip())

[   0.0s] 5f6c5b21-ffe9-48ee-9104-8d031ac2a159: processing (stage=text_extraction)
[   6.2s] 5f6c5b21-ffe9-48ee-9104-8d031ac2a159: completed (stage=text_extraction)
{'extractor': 'mineru', 'backend': None, 'mineru_version': None, 'text_characters': 32677}

Preview:
 # BRENNPUNKT LATEINAMERIKA

POLITIK · WIRTSCHAFT · GESELLSCHAFT

INSTITUT FÜR IBEROAMERIKA-KUNDE HAMBURG

Nummer 12

30. Juni 2000

ISSN 1437-6148

# Die politische Krise in Peru: Festsetzung des Fujimorismo und Polarisierung des Landes

Andreas Steinhauf

Von den umstrittensten und schmutzigsten Wahlen in der Geschichte Perus ist die Rede. Am 28. Mai 2000 wurde der amtierende Präsident Alberto Fujimori von der Obersten Wahlbehörde zum Sieger der Stichwahl erklärt, zu der sein Kontrahent Alejandro Toledo bereits nicht mehr angetreten war und stattdessen die Bevölkerung zum Boykott aufgerufen hatte. Damit wird er am 28. Juli, wenn verfassungsgemäß der neue Präsident vereidigt wird, seine nunm


## Step 4 — Extract raw bibliography entries with Qwen

`POST /extract/references` accepts Markdown as JSON. The `full_text` method sends the document to the configured medium-intelligence model (`Qwen3.6-27B-FP8` by default). Temperature `0.0` makes this validation run as deterministic as the serving stack permits.

In [7]:
submit_extraction_response = session.post(
    f"{API_BASE}/extract/references",
    params={"method": "full_text", "temperature": 0.0},
    json={"text": extracted_text},
    timeout=HTTP_TIMEOUT,
)

extraction_job = response_json(submit_extraction_response)
print(extraction_job)

{'job_id': 'c6f4a304-4669-48b3-9a66-71a04ec2f948', 'status': 'queued', 'created_at': '2026-07-22T16:04:23.862566', 'message': 'Reference extraction job enqueued'}


In [8]:
extraction_result = wait_for_job(extraction_job["job_id"])
raw_references = extraction_result.get("references", [])

assert raw_references, "Qwen returned no references for a document with a bibliography"
assert extraction_result.get("count") == len(raw_references)
assert all(isinstance(item, str) and item.strip() for item in raw_references)

print(f"Extracted {len(raw_references)} bibliography entries:")
for index, reference in enumerate(raw_references[:10], start=1):
    print(f"{index:>2}. {reference[:300]}")

[   0.0s] c6f4a304-4669-48b3-9a66-71a04ec2f948: processing (stage=reference_extraction)
[   4.1s] c6f4a304-4669-48b3-9a66-71a04ec2f948: completed (stage=reference_extraction)
Extracted 8 bibliography entries:
 1. Resumen Semanal, DESCO, Lima: http://www.desco.org.pe/rs-in.HTM
 2. Que Hacer, DESCO, Lima: http://www.desco.org.pe/qh/qh-in.htm#OH
 3. Caretas: http://www.caretas.com.pe/
 4. El Comercio: http://www.elcomercioperu.com
 5. Eigenen Beobachtungen vom 8. - 16 Juni 2000 in Lima
 6. Ingolf Dietrich. Die Koka- und Kokainwirtschaft Perus. Frankfurt/M.: Vervuert 1998, 314 S., ISBN 3-89354-247-7, Band 48
 7. Peter Thiery. Transformation in Chile. Institutioneller Wandel, Entwicklung und Demokratie 1973-1996. Frankfurt/M.: Vervuert 2000, ca. 354 S., Band 51
 8. Judith Schultz. Präsidentielle Demokratien in Lateinamerika. Eine Untersuchung der präsidentiellen Regierungssysteme von Costa Rica und Venezuela. Frankfurt/M.: Vervuert 2000, ca. 490 S., Band 52


## Step 5 — Parse the entries into structured records

`POST /parse/references` accepts a JSON list of strings. With `parser=llm`, Qwen returns fields such as authors, title, year, publication venue, volume, issue, and identifiers when they are present in the source citation.

In [9]:
submit_parse_response = session.post(
    f"{API_BASE}/parse/references",
    params={"parser": "llm", "temperature": 0.0},
    json={"references": raw_references},
    timeout=HTTP_TIMEOUT,
)

parse_job = response_json(submit_parse_response)
print(parse_job)

{'job_id': '262d39fd-dafa-433e-8a14-7e49e49641b0', 'status': 'queued', 'created_at': '2026-07-22T16:04:37.352586', 'message': 'Reference parsing job enqueued'}


In [10]:
parse_result = wait_for_job(parse_job["job_id"])
structured_references = parse_result.get("references", [])

assert parse_result.get("parser") == "llm", parse_result
assert parse_result.get("count") == len(structured_references)
assert len(structured_references) == len(raw_references)
assert all(isinstance(item, dict) for item in structured_references)

print(f"Parsed {len(structured_references)} structured references.")
print(json.dumps(structured_references[:2], indent=2, ensure_ascii=False)[:4_000])

[   0.0s] 262d39fd-dafa-433e-8a14-7e49e49641b0: processing (stage=reference_parsing)
[  14.5s] 262d39fd-dafa-433e-8a14-7e49e49641b0: completed (stage=reference_parsing)
Parsed 8 structured references.
[
  {
    "full_title": null,
    "journal_title": null,
    "authors": null,
    "editors": null,
    "publisher": "DESCO",
    "translator": null,
    "publication_place": "Lima",
    "publication_year": null,
    "publication_date_raw": null,
    "identifiers": [
      {
        "scheme": "URL",
        "value": "http://www.desco.org.pe/rs-in.HTM",
        "normalized": null
      }
    ],
    "ref_type": null,
    "raw": {},
    "volume": null,
    "issue": null,
    "pages": null,
    "cited_range": null,
    "footnote_number": null
  },
  {
    "full_title": null,
    "journal_title": null,
    "authors": null,
    "editors": null,
    "publisher": "DESCO",
    "translator": null,
    "publication_place": "Lima",
    "publication_year": null,
    "publication_date_raw": null,
    "i

## Checks — verify the error contract

A missing job must return HTTP 404. In production code, use `response_json` or equivalent handling so failed jobs and invalid IDs cannot be mistaken for empty results.

In [11]:
missing_job_id = str(uuid.uuid4())
missing_response = session.get(
    f"{API_BASE}/jobs/{missing_job_id}/status", timeout=HTTP_TIMEOUT
)
print({"status_code": missing_response.status_code, "body": missing_response.json()})
assert missing_response.status_code == 404

{'status_code': 404, 'body': {'detail': 'Job 1e71d90c-4590-4006-86a2-2ea9a9172b67 not found'}}


## Checks — final end-to-end verdict

This cell deliberately contains assertions, not just display logic. If it prints `PASS`, the tested PDF completed every supported stage and all required outputs were persisted.

In [12]:
end_to_end_checks = {
    "api_healthy": health["status"] == "healthy",
    "mineru_text_returned": len(extracted_text) > 1_000,
    "raw_references_returned": len(raw_references) > 0,
    "all_references_parsed": len(structured_references) == len(raw_references),
    "missing_job_returns_404": missing_response.status_code == 404,
}
assert all(end_to_end_checks.values()), end_to_end_checks

verdict = {
    "verdict": "PASS",
    "pdf": PDF_PATH.name,
    "text_characters": len(extracted_text),
    "raw_reference_count": len(raw_references),
    "structured_reference_count": len(structured_references),
    "jobs": {
        "text_extraction": text_job["job_id"],
        "reference_extraction": extraction_job["job_id"],
        "reference_parsing": parse_job["job_id"],
    },
}
print(json.dumps(verdict, indent=2))

{
  "verdict": "PASS",
  "pdf": "44404.pdf",
  "text_characters": 32677,
  "raw_reference_count": 8,
  "structured_reference_count": 8,
  "jobs": {
    "text_extraction": "5f6c5b21-ffe9-48ee-9104-8d031ac2a159",
    "reference_extraction": "c6f4a304-4669-48b3-9a66-71a04ec2f948",
    "reference_parsing": "262d39fd-dafa-433e-8a14-7e49e49641b0"
  }
}


## Equivalent command-line workflow

The first stage can be submitted with curl:

```bash
curl -F "file=@/absolute/path/paper.pdf;type=application/pdf" \
  "http://localhost:8001/extract/text?extractor=mineru&markdown=true"
```

Poll and retrieve any returned job ID:

```bash
curl http://localhost:8001/jobs/JOB_ID/status
curl http://localhost:8001/jobs/JOB_ID
```

For the JSON stages, write the previous response to a file or use `jq` to build the next request. Python is usually clearer for the complete chain because extracted document text can be large.

## API quick reference

| Method | Path | Queue | Input | Result |
|---|---|---|---|---|
| `GET` | `/health` | — | — | service health |
| `POST` | `/extract/text` | `default` | multipart PDF | Markdown text |
| `POST` | `/extract/references` | `llm-tasks` | `{"text": "..."}` | raw citation strings |
| `POST` | `/parse/references` | `llm-tasks` or `default` | `{"references": [...]}` | structured records |
| `GET` | `/jobs/{job_id}/status` | — | job ID | status and error metadata |
| `GET` | `/jobs/{job_id}` | — | completed job ID | persisted result |

Interactive OpenAPI documentation is available at [`http://localhost:8001/docs`](http://localhost:8001/docs).

## Next steps and operations

Use a different PDF by setting `CITATION_INDEX_PDF` before execution. For application integration, reuse the polling pattern but persist job IDs so work can resume after a client restart. Scale workers according to queue pressure rather than relying on a fixed concurrency claim.

Useful Docker commands:

```bash
docker compose logs -f api worker-default worker-llm
docker compose ps
docker compose down
```